<a href="https://colab.research.google.com/github/mohanad-abdalwahab/-/blob/main/%D8%A8%D9%8A%D8%A7%D9%86%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D9%88%D8%B8%D9%81%D9%8A%D9%86%20%D9%84%D9%84%D8%AA%D9%86%D8%B8%D9%8A%D9%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import glob
import re

# ===============================
# 1) قراءة ملفات الموظفين
# ===============================
hr_files = glob.glob("employees_*.xlsx")  # عدل الاسم حسب ملفاتك

if len(hr_files) == 0:
    raise ValueError("❌ لم يتم العثور على ملفات الموظفين")

all_hr = []

for file in hr_files:
    year = re.findall(r'\d{4}', file)[0]
    df = pd.read_excel(file)
    df["Year"] = int(year)
    all_hr.append(df)

hr = pd.concat(all_hr, ignore_index=True)

# ===============================
# 2) تنظيف الأعمدة
# ===============================
hr.columns = hr.columns.str.strip()

# أسماء الأعمدة (عدل حسب ملفك إذا اختلفت)
rename_map = {
    "رقم الموظف": "employee_id",
    "الجنس": "gender",
    "الجنسية": "nationality",
    "الإدارة": "department",
    "تاريخ الميلاد": "birth_date",
    "تاريخ التعيين": "hire_date",
    "نوع الإجراء": "action_type"
}

hr.rename(columns=rename_map, inplace=True)

# ===============================
# 3) حساب العمر
# ===============================
hr["birth_date"] = pd.to_datetime(hr["birth_date"], errors="coerce")
hr["age"] = hr["Year"] - hr["birth_date"].dt.year

hr.loc[(hr["age"] < 18) | (hr["age"] > 65), "age"] = np.nan

# ===============================
# 4) توحيد القيم
# ===============================
hr["gender"] = hr["gender"].replace({
    "ذكر": "Male",
    "أنثى": "Female"
})

hr["nationality_group"] = np.where(
    hr["nationality"] == "قطري",
    "Qatari",
    "Non_Qatari"
)

# ===============================
# 5) مؤشرات الأحداث
# ===============================
def contains(text, keyword):
    return text.fillna("").astype(str).str.contains(keyword)

hr["is_hire"] = contains(hr["action_type"], "تعيين").astype(int)
hr["is_promotion"] = contains(hr["action_type"], "ترقية").astype(int)
hr["is_transfer"] = contains(hr["action_type"], "نقل").astype(int)
hr["is_retirement"] = contains(hr["action_type"], "تقاعد").astype(int)

# ===============================
# 6) تجميع Department-Year
# ===============================
dept_year = hr.groupby(["department", "Year"]).agg(
    Total_Employees=("employee_id", "count"),
    Male_Count=("gender", lambda x: (x == "Male").sum()),
    Female_Count=("gender", lambda x: (x == "Female").sum()),
    Qatari_Count=("nationality_group", lambda x: (x == "Qatari").sum()),
    Non_Qatari_Count=("nationality_group", lambda x: (x == "Non_Qatari").sum()),
    Avg_Age=("age", "mean"),
    New_Hires_Count=("is_hire", "sum"),
    Promotions_Count=("is_promotion", "sum"),
    Transfers_Count=("is_transfer", "sum"),
    Retirements_Count=("is_retirement", "sum")
).reset_index()

# ===============================
# 7) قراءة ملف عبء العمل
# ===============================
workload = pd.read_excel("workload.xlsx")

workload.columns = workload.columns.str.strip()

# تحويل السنوات إلى صفوف
workload_long = workload.melt(
    id_vars=["الإدارة"],
    var_name="Year",
    value_name="Workload_Total"
)

workload_long.rename(columns={"الإدارة": "department"}, inplace=True)
workload_long["Year"] = pd.to_numeric(workload_long["Year"], errors="coerce")

# ===============================
# 8) دمج HR + Workload
# ===============================
final = pd.merge(dept_year, workload_long, on=["department", "Year"], how="left")

# ===============================
# 9) حساب الطلب والفجوة
# ===============================
final["Workforce_Demand"] = final["Workload_Total"] / 100  # عدل المعادلة حسب نظامك
final["Workforce_Gap"] = final["Workforce_Demand"] - final["Total_Employees"]

# ===============================
# 10) حفظ النتائج
# ===============================
final.to_csv("final_dataset.csv", index=False)
final.to_excel("final_dataset.xlsx", index=False)

print("✅ تم إنشاء dataset النهائي بنجاح")
print(final.head())

ValueError: ❌ لم يتم العثور على ملفات الموظفين

In [ ]:
from google.colab import drive
drive.mount('/content/drive')